# Forecasting Exchange Rates using Time Series Analysis
This notebook uses ARIMA and Exponential Smoothing models to forecast exchange rates.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load Dataset
df = pd.read_csv("exchange_rate (1).csv")

# Display first 5 rows
df.head()


In [ ]:
# Dataset Information
df.info()


In [ ]:
# Rename Columns
df.columns = ['Date', 'Exchange_Rate']


In [ ]:
# Convert Date Column
df['Date'] = pd.to_datetime(df['Date'])


In [ ]:
# Set Date as Index
df.set_index('Date', inplace=True)


In [ ]:
# Check Missing Values
df.isnull().sum()


In [ ]:
# Handle Missing Values
df.fillna(method='ffill', inplace=True)


In [ ]:
# Plot Time Series
plt.figure(figsize=(12,6))

plt.plot(df['Exchange_Rate'])

plt.title("Exchange Rate Time Series")
plt.xlabel("Date")
plt.ylabel("Exchange Rate")

plt.show()


In [ ]:
# Rolling Mean and Rolling Std
rolling_mean = df['Exchange_Rate'].rolling(window=12).mean()
rolling_std = df['Exchange_Rate'].rolling(window=12).std()

plt.figure(figsize=(12,6))

plt.plot(df['Exchange_Rate'], label='Original')
plt.plot(rolling_mean, label='Rolling Mean')
plt.plot(rolling_std, label='Rolling Std')

plt.legend()
plt.title("Rolling Statistics")

plt.show()


In [ ]:
# ACF Plot
plot_acf(df['Exchange_Rate'])
plt.show()


In [ ]:
# PACF Plot
plot_pacf(df['Exchange_Rate'])
plt.show()


In [ ]:
# Train Test Split
train_size = int(len(df) * 0.8)

train = df.iloc[:train_size]
test = df.iloc[train_size:]


In [ ]:
# ARIMA Model
model_arima = ARIMA(
    train['Exchange_Rate'],
    order=(1,1,1)
)

model_arima_fit = model_arima.fit()

print(model_arima_fit.summary())


In [ ]:
# ARIMA Forecast
forecast_arima = model_arima_fit.forecast(
    steps=len(test)
)


In [ ]:
# Plot ARIMA Forecast
plt.figure(figsize=(12,6))

plt.plot(train.index, train['Exchange_Rate'], label='Train')
plt.plot(test.index, test['Exchange_Rate'], label='Test')
plt.plot(test.index, forecast_arima, label='ARIMA Forecast')

plt.legend()
plt.title("ARIMA Forecast")

plt.show()


In [ ]:
# Exponential Smoothing Model
model_es = ExponentialSmoothing(
    train['Exchange_Rate'],
    trend='add',
    seasonal=None
)

model_es_fit = model_es.fit()


In [ ]:
# Exponential Smoothing Forecast
forecast_es = model_es_fit.forecast(
    len(test)
)


In [ ]:
# Plot Exponential Smoothing Forecast
plt.figure(figsize=(12,6))

plt.plot(train.index, train['Exchange_Rate'], label='Train')
plt.plot(test.index, test['Exchange_Rate'], label='Test')
plt.plot(test.index, forecast_es, label='ES Forecast')

plt.legend()
plt.title("Exponential Smoothing Forecast")

plt.show()


In [ ]:
# MAE Calculation
mae_arima = mean_absolute_error(
    test['Exchange_Rate'],
    forecast_arima
)

mae_es = mean_absolute_error(
    test['Exchange_Rate'],
    forecast_es
)

print("ARIMA MAE:", mae_arima)
print("Exponential Smoothing MAE:", mae_es)


In [ ]:
# RMSE Calculation
rmse_arima = np.sqrt(mean_squared_error(
    test['Exchange_Rate'],
    forecast_arima
))

rmse_es = np.sqrt(mean_squared_error(
    test['Exchange_Rate'],
    forecast_es
))

print("ARIMA RMSE:", rmse_arima)
print("Exponential Smoothing RMSE:", rmse_es)


In [ ]:
# MAPE Calculation
mape_arima = np.mean(
    np.abs(
        (test['Exchange_Rate'] - forecast_arima)
        / test['Exchange_Rate']
    )
) * 100

mape_es = np.mean(
    np.abs(
        (test['Exchange_Rate'] - forecast_es)
        / test['Exchange_Rate']
    )
) * 100

print("ARIMA MAPE:", mape_arima)
print("Exponential Smoothing MAPE:", mape_es)


In [ ]:
# Comparison Table
comparison = pd.DataFrame({
    'Model': ['ARIMA', 'Exponential Smoothing'],
    'MAE': [mae_arima, mae_es],
    'RMSE': [rmse_arima, rmse_es],
    'MAPE': [mape_arima, mape_es]
})

print(comparison)


# Conclusion

## Key Findings
- ARIMA and Exponential Smoothing models were applied successfully.
- ACF and PACF plots helped identify ARIMA parameters.
- Forecasting performance was evaluated using MAE, RMSE, and MAPE.

## Comparison
- ARIMA captures trends and dependencies in time series data.
- Exponential Smoothing works well for smooth and stable data.

## Final Insight
The better model depends on lower error metrics.
The model with lower MAE, RMSE, and MAPE provides more accurate forecasts.
